# Recommendation System

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity,euclidean_distances
from sklearn.preprocessing import MultiLabelBinarizer,MinMaxScaler

### Data Preprocessing:

In [62]:
df=pd.read_csv('anime.csv')
df.head(15)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
5,32935,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,"Comedy, Drama, School, Shounen, Sports",TV,10,9.15,93351
6,11061,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148,9.13,425855
7,820,Ginga Eiyuu Densetsu,"Drama, Military, Sci-Fi, Space",OVA,110,9.11,80679
8,15335,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,"Action, Comedy, Historical, Parody, Samurai, S...",Movie,1,9.10,72534
9,15417,Gintama&#039;: Enchousen,"Action, Comedy, Historical, Parody, Samurai, S...",TV,13,9.11,81109


In [29]:
df.shape

(12294, 7)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [31]:
df.nunique()

anime_id    12294
name        12292
genre        3264
type            6
episodes      187
rating        598
members      6706
dtype: int64

In [32]:
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

In [33]:
## missing values
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes    340
rating      230
members       0
dtype: int64

In [34]:
## handle missing value
df['genre']=df['genre'].fillna("Unknown")
df['type']=df['type'].fillna(df['type'].mode()[0])
df['rating']=df['rating'].fillna(df['rating'].mean())
df['episodes']=df['episodes'].fillna(df['episodes'].median())

In [35]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [36]:
## duplicates
df.duplicated().sum()

np.int64(0)

In [37]:
## mixed datatype
for col in df.columns:
    types = df[col].apply(type).nunique()
    if types > 1:
        print(col)

In [38]:
df.describe()

,anime_id,episodes,rating,members
count,12294.000000,12294.000000,12294.000000,1.229400e+04
mean,14058.221653,12.095412,6.473902,1.807134e+04
std,11455.294701,46.244062,1.017096,5.482068e+04
min,1.000000,1.000000,1.670000,5.000000e+00
25%,3484.250000,1.000000,5.900000,2.250000e+02
50%,10260.500000,2.000000,6.550000,1.550000e+03
75%,24794.500000,12.000000,7.170000,9.437000e+03
max,34527.000000,1818.000000,10.000000,1.013917e+06


In [39]:
## remove whitespaces,lowercase
df['name'] = df['name'].str.strip()
df['genre'] = df['genre'].str.strip().str.lower()  
df['type'] = df['type'].str.strip().str.lower() 

In [40]:
# Remove space before and after commas
df['genre'] = df['genre'].str.replace(" ,", ",", regex=False)
df['genre'] = df['genre'].str.replace(", ", ",", regex=False)

### Feature Extraction:

In [58]:
## feature used for computing similarity
feature=df[['genre','type','episodes','rating','members']].copy()
feature.head()

,genre,type,episodes,rating,members
0,"drama,romance,school,supernatural",movie,1.0,9.37,200630
1,"action,adventure,drama,fantasy,magic,military,...",tv,64.0,9.26,793665
2,"action,comedy,historical,parody,samurai,sci-fi...",tv,51.0,9.25,114262
3,"sci-fi,thriller",tv,24.0,9.17,673572
4,"action,comedy,historical,parody,samurai,sci-fi...",tv,51.0,9.16,151266


In [42]:
## convert genre into list
feature['genre_list'] = feature['genre'].apply(lambda x: x.split(","))

In [43]:
## genre encoding
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(feature['genre_list'])
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_)
feature = feature.join(genre_df)

In [44]:
## drop colum
feature = feature.drop(columns=['genre', 'genre_list'])

In [45]:
## type encoding
feature = pd.get_dummies(feature, columns=['type'], prefix='type',dtype=int)

In [46]:
## normalize numerical feature
min_max=MinMaxScaler()
feature[['episodes','rating','members']]=min_max.fit_transform(feature[['episodes','rating','members']])

In [47]:
feature.head()

,episodes,rating,members,action,adventure,cars,comedy,dementia,demons,drama,...,unknown,vampire,yaoi,yuri,type_movie,type_music,type_ona,type_ova,type_special,type_tv
0,0.000000,0.924370,0.197872,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,0.034673,0.911164,0.782770,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
2,0.027518,0.909964,0.112689,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0.012658,0.900360,0.664325,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0.027518,0.899160,0.149186,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### Recommendation System:

In [48]:
similarity=cosine_similarity(feature)
similarity.shape

(12294, 12294)

In [49]:
def recommend_anime(anime_name,threshold=0.5):
    name_lower = anime_name.lower()
    if name_lower not in df['name'].str.lower().values:
        print("Anime not found in the list.")
        return
    index = df[df['name'].str.lower() == name_lower].index[0]
    similarity_scores = list(enumerate(similarity[index]))
    similarity_scores = [(i, s) for i, s in similarity_scores if i != index]
    filtered = sorted([x for x in similarity_scores if x[1] >= threshold ],
    key=lambda x: x[1],reverse=True)
    if len(filtered) == 0:
        print(f"No anime found with similarity ≥ {threshold}.")
        return
    print(f"Anime similar to '{anime_name}' with similarity ≥ {threshold}:")
    print("-" * 50)
    for item in filtered:
        print(df.iloc[item[0]]['name'],"| genre:", df.iloc[item[0]]['genre'], "| score:", round(item[1], 3))

In [55]:
recommend_anime('Gintama°',threshold=0.9)

Anime similar to 'Gintama°' with similarity ≥ 0.9:
--------------------------------------------------
Gintama&#039; | genre: action,comedy,historical,parody,samurai,sci-fi,shounen | score: 1.0
Gintama&#039;: Enchousen | genre: action,comedy,historical,parody,samurai,sci-fi,shounen | score: 1.0
Gintama | genre: action,comedy,historical,parody,samurai,sci-fi,shounen | score: 0.997
Gintama (2017) | genre: action,comedy,historical,parody,samurai,sci-fi,shounen | score: 0.993


In [67]:
## list size based on threshold
recommend_anime('Steins;Gate',threshold=0.9)

Anime similar to 'Steins;Gate' with similarity ≥ 0.9:
--------------------------------------------------
Steins;Gate 0 | genre: Sci-Fi, Thriller | score: 0.945


In [66]:
recommend_anime('Steins;Gate',threshold=0.8)

Anime similar to 'Steins;Gate' with similarity ≥ 0.8:
--------------------------------------------------
Steins;Gate 0 | genre: Sci-Fi, Thriller | score: 0.945
Fireball Charming | genre: Sci-Fi | score: 0.806
Mirai Arise | genre: Sci-Fi | score: 0.801
Escha Chron | genre: Sci-Fi | score: 0.8


In [65]:
recommend_anime('Steins;Gate',threshold=0.7)

Anime similar to 'Steins;Gate' with similarity ≥ 0.7:
--------------------------------------------------
Steins;Gate 0 | genre: Sci-Fi, Thriller | score: 0.945
Fireball Charming | genre: Sci-Fi | score: 0.806
Mirai Arise | genre: Sci-Fi | score: 0.801
Escha Chron | genre: Sci-Fi | score: 0.8
Hoshi no Ko Poron | genre: Sci-Fi | score: 0.8
Yuusei Kamen | genre: Sci-Fi | score: 0.8
RoboDz | genre: Sci-Fi | score: 0.779
Hanoka | genre: Sci-Fi | score: 0.764
Zankyou no Terror | genre: Psychological, Thriller | score: 0.736
Steins;Gate Movie: Fuka Ryouiki no Déjà vu | genre: Sci-Fi, Thriller | score: 0.722
Steins;Gate: Oukoubakko no Poriomania | genre: Sci-Fi, Thriller | score: 0.717
Kiznaiver | genre: Drama, Sci-Fi | score: 0.714
Gankutsuou | genre: Drama, Mystery, Sci-Fi, Supernatural, Thriller | score: 0.712
No.6 | genre: Action, Sci-Fi | score: 0.711


### Interview Questions: